# 02 - Profiling e Qualidade Inicial dos Dados

## Objetivo

Realizar uma análise exploratória da qualidade dos dados armazenados na camada Bronze, identificando possíveis problemas que deverão ser tratados durante a construção da camada Silver.

A análise seguirá as dimensões de qualidade propostas no escopo do MVP:

- Completude;
- Consistência;
- Unicidade;
- Acurácia;
- Outliers.

O objetivo desta etapa não é alterar os dados, mas compreender sua estrutura e identificar problemas que possam impactar as análises posteriores.

In [0]:
df = spark.table("workspace.default.prf_acidentes_bronze")

In [0]:
print(f"Quantidade de registros: {df.count()}")
print(f"Quantidade de colunas: {len(df.columns)}")

Quantidade de registros: 342624
Quantidade de colunas: 32


In [0]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- data_inversa: string (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- horario: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- br: string (nullable = true)
 |-- km: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- causa_acidente: string (nullable = true)
 |-- tipo_acidente: string (nullable = true)
 |-- classificacao_acidente: string (nullable = true)
 |-- fase_dia: string (nullable = true)
 |-- sentido_via: string (nullable = true)
 |-- condicao_metereologica: string (nullable = true)
 |-- tipo_pista: string (nullable = true)
 |-- tracado_via: string (nullable = true)
 |-- uso_solo: string (nullable = true)
 |-- pessoas: string (nullable = true)
 |-- mortos: string (nullable = true)
 |-- feridos_leves: string (nullable = true)
 |-- feridos_graves: string (nullable = true)
 |-- ilesos: string (nullable = true)
 |-- ignorados: string (nullable = true)
 |-- feridos: string (nullable = true)


In [0]:
from pyspark.sql import functions as F

In [0]:
total_registros = df.count()

nulos = df.select([
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
])

display(nulos)

id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano_arquivo,arquivo_origem
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
resultado_nulos = []

for coluna in df.columns:
    qtd_nulos = df.filter(
        F.col(coluna).isNull() |
        (F.trim(F.col(coluna).cast("string")) == "")
    ).count()

    percentual = (qtd_nulos / total_registros) * 100

    resultado_nulos.append(
        (coluna, qtd_nulos, percentual)
    )

In [0]:
df_nulos = spark.createDataFrame(
    resultado_nulos,
    ["coluna", "qtd_nulos", "percentual_nulos"]
)

display(
    df_nulos.orderBy(
        F.desc("percentual_nulos")
    )
)

coluna,qtd_nulos,percentual_nulos
id,0,0.0
data_inversa,0,0.0
dia_semana,0,0.0
horario,0,0.0
uf,0,0.0
br,0,0.0
km,0,0.0
municipio,0,0.0
causa_acidente,0,0.0
tipo_acidente,0,0.0


### Análise inicial de completude

A verificação inicial de valores nulos e campos vazios não identificou ocorrências de `NULL` ou strings vazias nas 32 colunas da camada Bronze.

Entretanto, a ausência de valores tecnicamente nulos não garante que todos os atributos estejam completamente preenchidos. Bases públicas podem utilizar categorias textuais como `Não Informado`, `Ignorado`, `NA` ou outras expressões para representar informações ausentes ou não disponíveis.

Por esse motivo, a análise de completude será complementada pela investigação dos valores categóricos presentes em cada atributo.

In [0]:
marcadores_ausencia = [
    "NA",
    "N/A",
    "NULL",
    "IGNORADO",
    "NÃO INFORMADO",
    "NAO INFORMADO",
    "SEM INFORMAÇÃO",
    "SEM INFORMACAO"
]

In [0]:
resultado_ausencia = []

for coluna in df.columns:
    for marcador in marcadores_ausencia:
        quantidade = df.filter(
            F.upper(F.trim(F.col(coluna).cast("string"))) == marcador
        ).count()

        if quantidade > 0:
            resultado_ausencia.append(
                (coluna, marcador, quantidade)
            )

df_ausencia = spark.createDataFrame(
    resultado_ausencia,
    ["coluna", "marcador", "quantidade"]
)

display(
    df_ausencia.orderBy(
        F.desc("quantidade")
    )
)

coluna,marcador,quantidade
condicao_metereologica,IGNORADO,4492
sentido_via,NÃO INFORMADO,883
uop,N/A,267
delegacia,N/A,112
regional,NA,15
delegacia,NA,15
uop,NA,15
regional,N/A,12
classificacao_acidente,NA,5


### Completude semântica dos dados

Embora a análise inicial não tenha identificado valores `NULL` ou campos vazios, foi realizada uma segunda verificação procurando marcadores textuais utilizados para representar ausência ou desconhecimento de informação.

Foram encontrados registros contendo valores como `IGNORADO`, `NÃO INFORMADO`, `NA` e `N/A`.

Os principais casos identificados foram:

- `condicao_metereologica`: 4.492 registros classificados como `IGNORADO`;
- `sentido_via`: 883 registros classificados como `NÃO INFORMADO`;
- `uop`: 267 registros com `N/A` e 15 registros com `NA`;
- `delegacia`: 112 registros com `N/A` e 15 registros com `NA`;
- `regional`: 12 registros com `N/A` e 15 registros com `NA`;
- `classificacao_acidente`: 5 registros com `NA`.

Essa análise demonstra que a ausência de valores nulos técnicos não significa necessariamente completude dos dados, pois informações ausentes podem estar representadas por categorias textuais.

Os valores identificados serão analisados individualmente antes da definição das regras de tratamento da camada Silver.

In [0]:
df.groupBy("classificacao_acidente") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(truncate=False)

+----------------------+------+
|classificacao_acidente|count |
+----------------------+------+
|Com Vítimas Feridas   |260924|
|Sem Vítimas           |57077 |
|Com Vítimas Fatais    |24618 |
|NA                    |5     |
+----------------------+------+



In [0]:
display(
    df.filter(
        F.upper(F.trim(F.col("classificacao_acidente"))) == "NA"
    )
)

id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano_arquivo,arquivo_origem
652519,2025-01-01,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NA,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,2025,datatran2025.csv
571789,2024-01-01,segunda-feira,03:56:00,ES,101,38,CONCEICAO DA BARRA,Ultrapassagem Indevida,Colisão lateral sentido oposto,NA,Plena Noite,Crescente,Céu Claro,Simples,Reta,Não,3,0,0,1,1,1,1,3,-18.48261,-39.92379,SPRF-ES,DEL04-ES,UOP02-DEL04-ES,2024,datatran2024.csv
496590,2023-01-01,domingo,01:40:00,MT,163,1112,GUARANTA DO NORTE,Reação tardia ou ineficiente do condutor,Tombamento,NA,Plena Noite,Crescente,Ignorado,Simples,Curva;Declive,Não,2,0,0,1,0,2,1,3,"-9,70020602","-54,87588757",SPRF-MT,DEL06-MT,UOP03-DEL06-MT,2023,datatran2023.csv
331804,2021-01-01,sexta-feira,08:05:00,AM,174,937,MANAUS,Reação tardia ou ineficiente do condutor,Colisão traseira,NA,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,7,0,1,0,5,1,1,5,"-2,508068","-60,036434",SPRF-AM,DEL01-AM,UOP01-DEL01-AM,2021,datatran2021.csv
405151,2022-01-01,sábado,01:35:00,PI,316,415,MARCOLANDIA,Ingestão de álcool pelo condutor,Colisão traseira,NA,Plena Noite,Decrescente,Nublado,Simples,Reta,Sim,3,0,1,0,1,1,1,3,"-7,43280012","-40,68261908",SPRF-PI,DEL04-PI,UOP03-DEL04-PI,2022,datatran2022.csv


In [0]:
df.groupBy("condicao_metereologica") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(truncate=False)

+----------------------+------+
|condicao_metereologica|count |
+----------------------+------+
|Céu Claro             |214012|
|Nublado               |54014 |
|Chuva                 |34396 |
|Sol                   |20142 |
|Garoa/Chuvisco        |12118 |
|Ignorado              |4492  |
|Nevoeiro/Neblina      |2838  |
|Vento                 |593   |
|Granizo               |11    |
|Neve                  |8     |
+----------------------+------+



In [0]:
df.groupBy("sentido_via") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(truncate=False)

+-------------+------+
|sentido_via  |count |
+-------------+------+
|Crescente    |183409|
|Decrescente  |158332|
|Não Informado|883   |
+-------------+------+



In [0]:
df.groupBy("tipo_pista") \
  .count() \
  .orderBy(F.desc("count")) \
  .show(truncate=False)

+----------+------+
|tipo_pista|count |
+----------+------+
|Simples   |167198|
|Dupla     |143585|
|Múltipla  |31841 |
+----------+------+



## Análise de Unicidade

Nesta etapa será verificado se existem registros duplicados na camada Bronze.

Como a base utilizada está agrupada por ocorrência, espera-se que o campo `id` identifique de forma única cada acidente registrado.

Serão avaliadas duas situações:

- duplicidade do identificador `id`;
- duplicidade completa de registros.

In [0]:
duplicados_id = (
    df.groupBy("id")
      .count()
      .filter(F.col("count") > 1)
)

print(f"Quantidade de IDs duplicados: {duplicados_id.count()}")

Quantidade de IDs duplicados: 0


### Resultado da análise de unicidade

A verificação de unicidade do campo `id` não identificou registros duplicados.

Como a base utilizada está agrupada por ocorrência, era esperado que cada acidente possuísse um identificador único. O resultado confirma que não existem múltiplos registros com o mesmo `id` no conjunto analisado.

Dessa forma, não será necessário aplicar tratamento de duplicidade baseado no identificador da ocorrência na camada Silver.

## Análise de Consistência e Acurácia

Nesta etapa serão avaliados padrões e regras esperadas para os atributos da base.

O objetivo é identificar valores que, embora não sejam nulos, possam apresentar formato inválido, inconsistência ou valores incompatíveis com o contexto esperado.

A análise será realizada em grupos:

- datas e horários;
- campos geográficos;
- campos numéricos;
- campos categóricos;
- coerência lógica entre atributos.

In [0]:
from pyspark.sql import functions as F

df_formatos_data = (
    df.withColumn(
        "formato_data",
        F.when(
            F.col("data_inversa").rlike(r"^\d{4}-\d{2}-\d{2}$"),
            "yyyy-MM-dd"
        )
        .when(
            F.col("data_inversa").rlike(r"^\d{2}/\d{2}/\d{4}$"),
            "dd/MM/yyyy"
        )
        .otherwise("outro")
    )
)

display(
    df_formatos_data
    .groupBy("formato_data")
    .count()
    .orderBy(F.desc("count"))
)

formato_data,count
yyyy-MM-dd,342624


In [0]:
df_teste_data = df.withColumn(
    "data_convertida",
    F.to_date(F.col("data_inversa"), "yyyy-MM-dd")
)

datas_invalidas = df_teste_data.filter(
    F.col("data_convertida").isNull()
)

print(f"Datas inválidas: {datas_invalidas.count()}")

Datas inválidas: 0


In [0]:
datas_ano_inconsistente = df_teste_data.filter(
    F.year(F.col("data_convertida")) != F.col("ano_arquivo")
)

print(
    f"Registros com ano da data diferente do arquivo de origem: "
    f"{datas_ano_inconsistente.count()}"
)

Registros com ano da data diferente do arquivo de origem: 0


In [0]:
df_teste_horario = df.withColumn(
    "horario_convertido",
    F.to_timestamp(F.col("horario"), "HH:mm:ss")
)

horarios_invalidos = df_teste_horario.filter(
    F.col("horario_convertido").isNull()
)

print(f"Horários inválidos: {horarios_invalidos.count()}")

Horários inválidos: 0


In [0]:
display(
    df_teste_horario.select(
        F.min("horario_convertido").alias("horario_minimo"),
        F.max("horario_convertido").alias("horario_maximo")
    )
)

horario_minimo,horario_maximo
1970-01-01T00:00:00.000Z,1970-01-01T23:59:00.000Z


### Consistência das datas e horários

Foi realizada uma validação dos campos `data_inversa` e `horario`.

Todos os 342.624 registros da coluna `data_inversa` apresentaram o formato `yyyy-MM-dd`, não sendo identificadas inconsistências de formatação.

A conversão temporária da coluna para o tipo `date` não gerou valores inválidos.

Também foi verificada a compatibilidade entre o ano presente em `data_inversa` e a coluna `ano_arquivo`, utilizada para registrar a origem dos registros.

O campo `horario` foi validado por meio de conversão temporária utilizando o padrão `HH:mm:ss`, permitindo verificar a existência de horários inválidos.

Esses testes servirão de base para a conversão definitiva dos campos de data e horário na camada Silver.

### Consistência dos campos geográficos e rodoviários

Nesta etapa serão avaliados os campos relacionados à localização do acidente e à identificação da rodovia.

Serão verificadas:

- validade das siglas de UF;
- presença de valores inválidos em rodovia (`br`);
- consistência do campo `km`;
- validade das coordenadas de latitude e longitude.

O objetivo é identificar valores incompatíveis com os domínios esperados antes da construção da camada Silver.

In [0]:
ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
]

ufs_invalidas = (
    df.filter(~F.col("uf").isin(ufs_validas))
      .groupBy("uf")
      .count()
      .orderBy(F.desc("count"))
)

display(ufs_invalidas)

uf,count


In [0]:
display(
    df.groupBy("uf")
      .count()
      .orderBy(F.desc("count"))
)

uf,count
MG,44502
SC,39849
PR,37055
RJ,27669
RS,24511
SP,23005
BA,18779
GO,15940
PE,14549
ES,12083


In [0]:
br_nao_numerica = (
    df.filter(
        ~F.col("br").rlike(r"^\d+$")
    )
    .groupBy("br")
    .count()
    .orderBy(F.desc("count"))
)

display(br_nao_numerica)

br,count


In [0]:
display(
    df.select(
        F.min(F.col("br").cast("int")).alias("br_minima"),
        F.max(F.col("br").cast("int")).alias("br_maxima")
    )
)

br_minima,br_maxima
0,498


In [0]:
display(
    df.select("km")
      .distinct()
      .orderBy("km")
      .limit(50)
)

km
0
"0,1"
"0,2"
"0,3"
"0,4"
"0,5"
"0,6"
"0,7"
"0,8"
"0,9"


In [0]:
df_teste_km = df.withColumn(
    "km_convertido",
    F.regexp_replace(F.col("km"), ",", ".").cast("double")
)

km_invalidos = df_teste_km.filter(
    F.col("km_convertido").isNull()
)

print(f"Valores de km que não puderam ser convertidos: {km_invalidos.count()}")

Valores de km que não puderam ser convertidos: 0


In [0]:
display(
    df_teste_km.select(
        F.min("km_convertido").alias("km_minimo"),
        F.max("km_convertido").alias("km_maximo")
    )
)

km_minimo,km_maximo
0.0,1470.0


In [0]:
df_geo = (
    df.withColumn(
        "latitude_num",
        F.regexp_replace(F.col("latitude"), ",", ".").cast("double")
    )
    .withColumn(
        "longitude_num",
        F.regexp_replace(F.col("longitude"), ",", ".").cast("double")
    )
)

In [0]:
geo_nao_convertidos = df_geo.filter(
    F.col("latitude_num").isNull() |
    F.col("longitude_num").isNull()
)

print(
    f"Coordenadas que não puderam ser convertidas: "
    f"{geo_nao_convertidos.count()}"
)

Coordenadas que não puderam ser convertidas: 0


In [0]:
coordenadas_invalidas = df_geo.filter(
    (F.col("latitude_num") < -90) |
    (F.col("latitude_num") > 90) |
    (F.col("longitude_num") < -180) |
    (F.col("longitude_num") > 180)
)

print(
    f"Coordenadas fora dos limites geográficos válidos: "
    f"{coordenadas_invalidas.count()}"
)

Coordenadas fora dos limites geográficos válidos: 0


In [0]:
display(
    df_geo.select(
        F.min("latitude_num").alias("latitude_min"),
        F.max("latitude_num").alias("latitude_max"),
        F.min("longitude_num").alias("longitude_min"),
        F.max("longitude_num").alias("longitude_max")
    )
)

latitude_min,latitude_max,longitude_min,longitude_max
-33.689819,4.47628443,-72.665988,-32.406822


### Resultado da consistência dos campos geográficos e rodoviários

Foram realizadas validações sobre os campos `uf`, `br`, `km`, `latitude` e `longitude`.

As siglas de UF apresentaram valores compatíveis com o domínio esperado das unidades federativas brasileiras.

O campo `br` não apresentou valores não numéricos relevantes para o conjunto analisado.

A coluna `km` pôde ser convertida para formato numérico após a substituição da vírgula decimal por ponto, sem geração de valores inválidos.

As colunas `latitude` e `longitude` também puderam ser convertidas para formato numérico e não apresentaram coordenadas fora dos limites geográficos válidos.

Dessa forma, não foram identificados problemas de consistência relevantes nesses atributos. Na camada Silver, esses campos serão convertidos para tipos numéricos apropriados.

In [0]:
colunas_numericas = [
    "pessoas",
    "mortos",
    "feridos_leves",
    "feridos_graves",
    "ilesos",
    "ignorados",
    "feridos",
    "veiculos"
]

for coluna in colunas_numericas:
    invalidos = df.filter(
        F.col(coluna).cast("int").isNull()
    ).count()

    print(f"{coluna}: {invalidos} valores não numéricos")

pessoas: 0 valores não numéricos
mortos: 0 valores não numéricos
feridos_leves: 0 valores não numéricos
feridos_graves: 0 valores não numéricos
ilesos: 0 valores não numéricos
ignorados: 0 valores não numéricos
feridos: 0 valores não numéricos
veiculos: 0 valores não numéricos


In [0]:
display(
    df.select([
        F.col(c).cast("int").alias(c)
        for c in colunas_numericas
    ]).summary()
)

summary,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos
count,342624,342624,342624,342624,342624,342624,342624,342624
mean,2.5733194405529094,0.08367189688988512,0.8648051508359017,0.2777797235453442,1.0354645325487999,0.39836088540207343,1.142584874381246,1.9865158307649202
stddev,2.169087019014003,0.3444691208222189,1.076105718844053,0.6187614190902969,1.7048702763741215,0.8490610497928278,1.1977629280557311,1.1195900251649427
min,1,0,0,0,0,0,0,1
25%,2,0,0,0,0,0,1,1
50%,2,0,1,0,1,0,1,2
75%,3,0,1,0,1,1,1,2
max,95,37,83,35,78,88,84,131


In [0]:
inconsistencias_logicas = df.filter(
    (F.col("mortos").cast("int") > F.col("pessoas").cast("int")) |
    (F.col("feridos").cast("int") > F.col("pessoas").cast("int")) |
    (
        (
            F.col("feridos_leves").cast("int") +
            F.col("feridos_graves").cast("int")
        ) > F.col("pessoas").cast("int")
    ) |
    (F.col("veiculos").cast("int") < 1)
)

print(
    f"Registros com inconsistências lógicas básicas: "
    f"{inconsistencias_logicas.count()}"
)

Registros com inconsistências lógicas básicas: 0


In [0]:
inconsistencia_feridos = df.filter(
    F.col("feridos").cast("int") != (
        F.col("feridos_leves").cast("int") +
        F.col("feridos_graves").cast("int")
    )
)

print(
    f"Registros em que feridos != leves + graves: "
    f"{inconsistencia_feridos.count()}"
)

Registros em que feridos != leves + graves: 0


### Validação dos campos numéricos

Os campos relacionados às quantidades de pessoas, vítimas e veículos foram convertidos temporariamente para valores inteiros e analisados por meio de estatísticas descritivas.

Todos os campos apresentaram 342.624 valores válidos, indicando que os registros puderam ser convertidos para formato numérico sem perda de informação.

Também não foram identificados valores negativos nas variáveis analisadas.

Foram observados valores máximos significativamente superiores aos valores típicos de algumas variáveis, como quantidade de pessoas e veículos envolvidos. Esses registros serão avaliados posteriormente na análise de outliers, pois valores extremos não necessariamente representam erros nos dados.

In [0]:
inconsistencias_logicas = df.filter(
    (F.col("mortos").cast("int") > F.col("pessoas").cast("int")) |
    (F.col("feridos").cast("int") > F.col("pessoas").cast("int")) |
    (
        (
            F.col("feridos_leves").cast("int") +
            F.col("feridos_graves").cast("int")
        ) > F.col("pessoas").cast("int")
    ) |
    (F.col("veiculos").cast("int") < 1)
)

print(
    f"Registros com inconsistências lógicas básicas: "
    f"{inconsistencias_logicas.count()}"
)

Registros com inconsistências lógicas básicas: 0


In [0]:
inconsistencia_feridos = df.filter(
    F.col("feridos").cast("int") != (
        F.col("feridos_leves").cast("int") +
        F.col("feridos_graves").cast("int")
    )
)

print(
    f"Registros em que feridos != leves + graves: "
    f"{inconsistencia_feridos.count()}"
)

Registros em que feridos != leves + graves: 0


## Análise dos Campos Categóricos

Nesta etapa serão avaliados os principais atributos categóricos da base.

O objetivo é verificar:

- quais categorias existem em cada campo;
- se existem valores com grafias ou formatos inconsistentes;
- se existem categorias utilizadas para representar ausência de informação;
- se os valores observados estão coerentes com o contexto do atributo.

Essa análise será utilizada posteriormente para definir regras de padronização na camada Silver.

In [0]:
colunas_categoricas = [
    "dia_semana",
    "causa_acidente",
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "sentido_via",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via",
    "uso_solo"
]

In [0]:
for coluna in colunas_categoricas:
    quantidade = df.select(coluna).distinct().count()
    print(f"{coluna}: {quantidade} categorias distintas")

dia_semana: 7 categorias distintas
causa_acidente: 76 categorias distintas
tipo_acidente: 18 categorias distintas
classificacao_acidente: 4 categorias distintas
fase_dia: 4 categorias distintas
sentido_via: 3 categorias distintas
condicao_metereologica: 10 categorias distintas
tipo_pista: 3 categorias distintas
tracado_via: 1214 categorias distintas
uso_solo: 2 categorias distintas


In [0]:
display(
    df.groupBy("dia_semana")
      .count()
      .orderBy(F.desc("count"))
)

dia_semana,count
domingo,56278
sábado,56111
sexta-feira,53009
segunda-feira,47208
quinta-feira,44343
quarta-feira,43392
terça-feira,42283


In [0]:
display(
    df.groupBy("classificacao_acidente")
      .count()
      .orderBy(F.desc("count"))
)

classificacao_acidente,count
Com Vítimas Feridas,260924
Sem Vítimas,57077
Com Vítimas Fatais,24618
NA,5


In [0]:
display(
    df.groupBy("fase_dia")
      .count()
      .orderBy(F.desc("count"))
)

fase_dia,count
Pleno dia,187621
Plena Noite,119581
Anoitecer,18821
Amanhecer,16601


In [0]:
display(
    df.groupBy("sentido_via")
      .count()
      .orderBy(F.desc("count"))
)

sentido_via,count
Crescente,183409
Decrescente,158332
Não Informado,883


In [0]:
display(
    df.groupBy("condicao_metereologica")
      .count()
      .orderBy(F.desc("count"))
)

condicao_metereologica,count
Céu Claro,214012
Nublado,54014
Chuva,34396
Sol,20142
Garoa/Chuvisco,12118
Ignorado,4492
Nevoeiro/Neblina,2838
Vento,593
Granizo,11
Neve,8


In [0]:
display(
    df.groupBy("tipo_pista")
      .count()
      .orderBy(F.desc("count"))
)

tipo_pista,count
Simples,167198
Dupla,143585
Múltipla,31841


In [0]:
display(
    df.groupBy("uso_solo")
      .count()
      .orderBy(F.desc("count"))
)

uso_solo,count
Não,194181
Sim,148443


In [0]:
display(
    df.groupBy("causa_acidente")
      .count()
      .orderBy(F.desc("count"))
)

causa_acidente,count
Reação tardia ou ineficiente do condutor,46901
Ausência de reação do condutor,44630
Acessar a via sem observar a presença dos outros veículos,31168
Velocidade Incompatível,24420
Condutor deixou de manter distância do veículo da frente,22601
Ingestão de álcool pelo condutor,20077
Manobra de mudança de faixa,19442
Demais falhas mecânicas ou elétricas,15126
Transitar na contramão,11145
Condutor Dormindo,10822


In [0]:
display(
    df.groupBy("tipo_acidente")
      .count()
      .orderBy(F.desc("count"))
)

tipo_acidente,count
Colisão traseira,65634
Saída de leito carroçável,52394
Colisão transversal,43190
Colisão lateral mesmo sentido,34602
Tombamento,29582
Colisão com objeto,24761
Colisão frontal,23045
Atropelamento de Pedestre,15313
Queda de ocupante de veículo,15251
Colisão lateral sentido oposto,9582


In [0]:
display(
    df.groupBy("tracado_via")
      .count()
      .orderBy(F.desc("count"))
)

tracado_via,count
Reta,196658
Curva,43460
Interseção de Vias,10544
Reta;Declive,7708
Reta;Aclive,6692
Declive,6448
Curva;Declive,5930
Aclive;Reta,5534
Rotatória,5426
Aclive,5396


In [0]:
for coluna in colunas_categoricas:
    distintos_original = df.select(coluna).distinct().count()

    distintos_trim = (
        df.select(F.trim(F.col(coluna)).alias(coluna))
          .distinct()
          .count()
    )

    if distintos_original != distintos_trim:
        print(
            f"{coluna}: possível inconsistência por espaços "
            f"({distintos_original} -> {distintos_trim})"
        )

In [0]:
for coluna in colunas_categoricas:
    distintos_original = df.select(coluna).distinct().count()

    distintos_padronizados = (
        df.select(
            F.upper(F.trim(F.col(coluna))).alias(coluna)
        )
        .distinct()
        .count()
    )

    if distintos_original != distintos_padronizados:
        print(
            f"{coluna}: possível inconsistência de capitalização "
            f"({distintos_original} -> {distintos_padronizados})"
        )

causa_acidente: possível inconsistência de capitalização (76 -> 75)


### Investigação de possível mudança de classificação em `tipo_acidente`

Durante a inspeção visual das categorias da coluna `tipo_acidente`, foram observados valores semelhantes relacionados a colisões laterais, como:

- `Colisão lateral`
- `Colisão lateral mesmo sentido`
- `Colisão lateral sentido oposto`

Como essas categorias podem representar tanto diferenças reais de classificação quanto uma possível mudança no padrão de registro ao longo dos anos, foi realizada uma verificação por `ano_arquivo`.

O objetivo é identificar se a categoria genérica `Colisão lateral` aparece predominantemente nos anos mais antigos e se, nos anos mais recentes, passa a ser substituída por classificações mais específicas.

Essa análise permite evitar uma padronização incorreta de categorias que podem ter significados distintos ou refletir mudanças no padrão de registro da fonte.

In [0]:
display(
    df.filter(
        F.col("tipo_acidente").contains("Colisão lateral")
    )
    .groupBy("ano_arquivo", "tipo_acidente")
    .count()
    .orderBy("ano_arquivo", "tipo_acidente")
)

ano_arquivo,tipo_acidente,count
2021,Colisão lateral,676
2021,Colisão lateral mesmo sentido,5572
2021,Colisão lateral sentido oposto,1609
2022,Colisão lateral mesmo sentido,6239
2022,Colisão lateral sentido oposto,1775
2023,Colisão lateral mesmo sentido,7004
2023,Colisão lateral sentido oposto,2018
2024,Colisão lateral mesmo sentido,7902
2024,Colisão lateral sentido oposto,2028
2025,Colisão lateral mesmo sentido,7885


In [0]:
display(
    df.filter(
        (F.col("ano_arquivo") == 2021) &
        (F.col("tipo_acidente") == "Colisão lateral")
    )
    .withColumn(
        "mes",
        F.month(F.to_date(F.col("data_inversa"), "yyyy-MM-dd"))
    )
    .groupBy("mes")
    .count()
    .orderBy("mes")
)

mes,count
1,376
2,300


### Resultado da investigação de `Colisão lateral`

A análise por ano mostrou que a categoria genérica `Colisão lateral` ocorre apenas em 2021, com 676 registros.

Ao detalhar esses registros por mês, verificou-se que todos estão concentrados nos dois primeiros meses do ano:

- janeiro de 2021: 376 registros;
- fevereiro de 2021: 300 registros.

A partir de março de 2021, a categoria genérica deixa de aparecer, permanecendo apenas as classificações mais específicas `Colisão lateral mesmo sentido` e `Colisão lateral sentido oposto`.

Esse comportamento sugere uma mudança ou refinamento no padrão de classificação da fonte durante o início de 2021.

Como não é possível determinar, a partir dos dados disponíveis, a qual das duas classificações específicas pertencem os registros originalmente classificados apenas como `Colisão lateral`, esses registros serão preservados como categoria própria.

Dessa forma, evita-se introduzir uma classificação que não está explicitamente presente na fonte original.

### Investigação de inconsistência em `causa_acidente`

Durante a análise das categorias, foi identificado que a coluna `causa_acidente` possui 76 valores distintos originalmente, mas 75 após padronização de maiúsculas/minúsculas.

Isso indica que pelo menos duas categorias diferem apenas pela forma de escrita. A seguir será identificada essa variação para definir a regra de padronização na camada Silver.

In [0]:
display(
    df.groupBy(
        F.upper(F.trim(F.col("causa_acidente"))).alias("causa_padronizada")
    )
    .agg(
        F.countDistinct("causa_acidente").alias("qtd_variacoes"),
        F.collect_set("causa_acidente").alias("variacoes"),
        F.count("*").alias("qtd_registros")
    )
    .filter(F.col("qtd_variacoes") > 1)
)

causa_padronizada,qtd_variacoes,variacoes,qtd_registros
TRANSITAR NO ACOSTAMENTO,2,"List(Transitar no Acostamento, Transitar no acostamento)",2080


### Investigação da coluna `tracado_via`

A coluna `tracado_via` apresentou 1.214 valores distintos, quantidade muito superior às demais variáveis categóricas.

Durante a inspeção dos dados, foi observado que alguns registros apresentam mais de uma característica de traçado no mesmo campo, separadas por ponto e vírgula.

Por esse motivo, será verificado quantos registros apresentam múltiplas características e quais combinações são mais frequentes.

In [0]:
registros_multiplos_tracado = df.filter(
    F.col("tracado_via").contains(";")
).count()

percentual_multiplos = (
    registros_multiplos_tracado / df.count()
) * 100

print(
    f"Registros com múltiplas características de traçado: "
    f"{registros_multiplos_tracado}"
)

print(
    f"Percentual: {percentual_multiplos:.2f}%"
)

Registros com múltiplas características de traçado: 67741
Percentual: 19.77%


In [0]:
display(
    df.groupBy("tracado_via")
      .count()
      .orderBy(F.desc("count"))
      .limit(30)
)

tracado_via,count
Reta,196658
Curva,43460
Interseção de Vias,10544
Reta;Declive,7708
Reta;Aclive,6692
Declive,6448
Curva;Declive,5930
Aclive;Reta,5534
Rotatória,5426
Aclive,5396


In [0]:
df_tracado_explodido = (
    df.select(
        F.explode(
            F.split(F.col("tracado_via"), ";")
        ).alias("caracteristica_tracado")
    )
    .withColumn(
        "caracteristica_tracado",
        F.trim(F.col("caracteristica_tracado"))
    )
)

display(
    df_tracado_explodido
    .groupBy("caracteristica_tracado")
    .count()
    .orderBy(F.desc("count"))
)

caracteristica_tracado,count
Reta,241869
Curva,62755
Declive,32735
Aclive,25085
Interseção de Vias,22802
Rotatória,8422
Retorno Regulamentado,7830
Em Obras,6656
Viaduto,5210
Ponte,3540


### Resultado da análise dos campos categóricos

A análise dos atributos categóricos identificou dois pontos relevantes para o processo de transformação.

#### Inconsistência em `causa_acidente`

A coluna `causa_acidente` apresentou 76 categorias distintas originalmente, mas 75 após padronização de maiúsculas e minúsculas.

Foi identificado que as categorias `Transitar no Acostamento` e `Transitar no acostamento` representam o mesmo valor, diferindo apenas pela capitalização do texto. Juntas, essas variações correspondem a 2.080 registros.

Essa inconsistência será corrigida na camada Silver por meio da padronização da categoria.

#### Estrutura multivalorada de `tracado_via`

A coluna `tracado_via` apresentou 1.214 valores distintos. A investigação mostrou que esse elevado número não representa 1.214 tipos diferentes de traçado, mas combinações de múltiplas características armazenadas no mesmo campo e separadas por ponto e vírgula.

Foram identificados 67.741 registros com múltiplas características de traçado, correspondendo a 19,77% do conjunto de dados.

Após a separação dessas combinações, foram identificadas 12 características individuais de traçado.

Dessa forma, o valor original de `tracado_via` será preservado para garantir rastreabilidade, porém a estrutura multivalorada será considerada posteriormente na modelagem para permitir análises individuais das características da via.

## Análise de Outliers

Nesta etapa serão investigados valores extremos nos principais campos numéricos da base.

O objetivo não é remover automaticamente registros muito altos, pois eventos de trânsito podem apresentar ocorrências legítimas envolvendo grandes quantidades de pessoas, vítimas ou veículos.

Os valores extremos serão inicialmente identificados e posteriormente avaliados quanto à sua plausibilidade no contexto dos dados.

In [0]:
colunas_outliers = [
    "pessoas",
    "mortos",
    "feridos_leves",
    "feridos_graves",
    "ilesos",
    "ignorados",
    "feridos",
    "veiculos"
]

In [0]:
resultado_outliers = []

for coluna in colunas_outliers:

    df_num = df.select(
        F.col(coluna).cast("double").alias(coluna)
    )

    q1, q3 = df_num.approxQuantile(
        coluna,
        [0.25, 0.75],
        0.01
    )

    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    qtd_outliers = df_num.filter(
        (F.col(coluna) < limite_inferior) |
        (F.col(coluna) > limite_superior)
    ).count()

    resultado_outliers.append(
        (
            coluna,
            q1,
            q3,
            iqr,
            limite_inferior,
            limite_superior,
            qtd_outliers
        )
    )

In [0]:
df_outliers = spark.createDataFrame(
    resultado_outliers,
    [
        "coluna",
        "q1",
        "q3",
        "iqr",
        "limite_inferior",
        "limite_superior",
        "qtd_outliers"
    ]
)

display(df_outliers)

coluna,q1,q3,iqr,limite_inferior,limite_superior,qtd_outliers
pessoas,2.0,3.0,1.0,0.5,4.5,29530
mortos,0.0,0.0,0.0,0.0,0.0,24619
feridos_leves,0.0,1.0,1.0,-1.5,2.5,14383
feridos_graves,0.0,0.0,0.0,0.0,0.0,77574
ilesos,0.0,1.0,1.0,-1.5,2.5,28515
ignorados,0.0,1.0,1.0,-1.5,2.5,9141
feridos,1.0,1.0,0.0,1.0,1.0,149906
veiculos,1.0,2.0,1.0,-0.5,3.5,24561


In [0]:
display(
    df.select(
        "id",
        "data_inversa",
        "uf",
        "municipio",
        "tipo_acidente",
        "pessoas",
        "mortos",
        "feridos",
        "veiculos"
    )
    .orderBy(F.col("pessoas").cast("int").desc())
    .limit(20)
)

id,data_inversa,uf,municipio,tipo_acidente,pessoas,mortos,feridos,veiculos
595429,2023-09-02,PR,BALSA NOVA,Colisão traseira,95,5,12,131
659555,2024-12-04,BA,OLINDINA,Colisão com objeto,93,5,84,5
651553,2024-12-27,GO,URUACU,Colisão lateral sentido oposto,80,2,0,2
716289,2025-09-05,MG,JUIZ DE FORA,Colisão traseira,76,4,38,2
456403,2022-03-30,BA,CORRENTINA,Tombamento,75,4,54,1
734818,2025-11-28,MG,FRANCISCO SA,Colisão traseira,73,0,1,4
536685,2023-07-18,GO,ANAPOLIS,Colisão traseira,73,1,27,3
673084,2025-04-12,SP,BARRA DO TURVO,Incêndio,71,0,0,1
404997,2021-12-31,BA,CORRENTINA,Colisão com objeto,70,4,20,2
474624,2022-07-22,MG,BURITIZEIRO,Saída de leito carroçável,70,0,14,1


In [0]:
display(
    df.select(
        "id",
        "data_inversa",
        "uf",
        "municipio",
        "tipo_acidente",
        "pessoas",
        "mortos",
        "feridos",
        "veiculos"
    )
    .orderBy(F.col("veiculos").cast("int").desc())
    .limit(20)
)

id,data_inversa,uf,municipio,tipo_acidente,pessoas,mortos,feridos,veiculos
595429,2023-09-02,PR,BALSA NOVA,Colisão traseira,95,5,12,131
707992,2025-07-07,GO,PADRE BERNARDO,Incêndio,2,0,0,82
748976,2025-04-06,SC,PALHOCA,Colisão com objeto,33,0,8,31
470887,2022-08-28,TO,WANDERLANDIA,Colisão traseira,5,1,1,28
633813,2024-10-10,PR,BALSA NOVA,Engavetamento,19,1,7,26
550853,2023-09-25,GO,ABADIANIA,Engavetamento,29,3,18,25
650541,2024-12-22,TO,AGUIARNOPOLIS,Queda de ocupante de veículo,30,14,1,24
389907,2021-09-17,MG,MONTE ALEGRE DE MINAS,Engavetamento,16,1,5,23
452058,2022-05-26,PB,SANTA RITA,Colisão traseira,32,0,17,22
704931,2025-07-15,ES,PEDRO CANARIO,Colisão frontal,11,1,8,21


### Interpretação dos campos `pessoas` e `veiculos`

Durante a análise dos valores extremos, foram observadas ocorrências em que a quantidade de veículos registrados é superior à quantidade de pessoas envolvidas.

Segundo o dicionário de dados da PRF:

- `pessoas` representa o total de pessoas envolvidas na ocorrência;
- `veiculos` representa o total de veículos envolvidos na ocorrência.

Entretanto, materiais analíticos da própria PRF indicam que uma parcela das pessoas ilesas envolvidas em acidentes pode não ser identificada nos registros do Boletim de Acidente de Trânsito (BAT).

Dessa forma, registros com quantidade de veículos superior à quantidade de pessoas não serão classificados automaticamente como inconsistentes.

Esses casos serão tratados como valores que exigem cautela de interpretação, preservando-se os registros originais na camada Silver, salvo identificação de evidência adicional de erro.

In [0]:
casos_mais_veiculos_que_pessoas = df.filter(
    F.col("veiculos").cast("int") > F.col("pessoas").cast("int")
)

qtd_mais_veiculos = casos_mais_veiculos_que_pessoas.count()
percentual_mais_veiculos = qtd_mais_veiculos / df.count() * 100

print(f"Registros com veículos > pessoas: {qtd_mais_veiculos}")
print(f"Percentual: {percentual_mais_veiculos:.2f}%")

Registros com veículos > pessoas: 14249
Percentual: 4.16%


In [0]:
display(
    casos_mais_veiculos_que_pessoas
    .select(
        "id",
        "data_inversa",
        "uf",
        "municipio",
        "tipo_acidente",
        "pessoas",
        "mortos",
        "feridos",
        "veiculos"
    )
    .orderBy(
        (F.col("veiculos").cast("int") - F.col("pessoas").cast("int")).desc()
    )
    .limit(30)
)

id,data_inversa,uf,municipio,tipo_acidente,pessoas,mortos,feridos,veiculos
707992,2025-07-07,GO,PADRE BERNARDO,Incêndio,2,0,0,82
595429,2023-09-02,PR,BALSA NOVA,Colisão traseira,95,5,12,131
470887,2022-08-28,TO,WANDERLANDIA,Colisão traseira,5,1,1,28
571340,2023-12-29,BA,JAGUAQUARA,Colisão traseira,4,0,2,19
520328,2023-05-01,BA,RIBEIRA DO POMBAL,Colisão lateral mesmo sentido,4,0,0,18
498307,2023-01-09,MG,GRAO MOGOL,Tombamento,4,0,1,17
685337,2025-04-12,MT,NOBRES,Incêndio,2,0,0,15
635337,2024-09-25,GO,MINEIROS,Tombamento,2,0,1,15
449169,2022-02-17,RJ,SEROPEDICA,Colisão traseira,4,1,1,16
524517,2023-05-21,MG,FRANCISCO SA,Colisão traseira,3,0,1,15


In [0]:
casos_veiculos_igual_pessoas = df.filter(
    F.col("veiculos").cast("int") == F.col("pessoas").cast("int")
)

qtd_igual = casos_veiculos_igual_pessoas.count()
percentual_igual = qtd_igual / df.count() * 100

print(f"Registros com veículos = pessoas: {qtd_igual}")
print(f"Percentual: {percentual_igual:.2f}%")

Registros com veículos = pessoas: 210356
Percentual: 61.40%


In [0]:
casos_mais_pessoas_que_veiculos = df.filter(
    F.col("pessoas").cast("int") > F.col("veiculos").cast("int")
)

qtd_mais_pessoas = casos_mais_pessoas_que_veiculos.count()
percentual_mais_pessoas = qtd_mais_pessoas / df.count() * 100

print(f"Registros com pessoas > veículos: {qtd_mais_pessoas}")
print(f"Percentual: {percentual_mais_pessoas:.2f}%")

Registros com pessoas > veículos: 118019
Percentual: 34.45%


### Resultado da análise de outliers

A análise estatística identificou valores extremos em variáveis como `pessoas`, `mortos`, `feridos` e `veiculos`.

Foram observadas ocorrências com números significativamente superiores ao comportamento típico da base, como acidentes envolvendo dezenas de pessoas ou veículos.

Esses registros não foram classificados automaticamente como erros, pois acidentes de grande porte, como engavetamentos ou ocorrências envolvendo ônibus, podem gerar valores elevados legítimos.

Também foram identificados casos em que a quantidade de veículos é superior à quantidade de pessoas registradas. Conforme a documentação da PRF, isso pode estar relacionado à forma de registro das pessoas envolvidas, especialmente quando há sub-registro de pessoas ilesas.

Por esse motivo, os valores extremos serão preservados na camada Silver. A análise de outliers será utilizada como mecanismo de avaliação e interpretação, e não como regra automática de exclusão de registros.

## Resumo da Análise de Qualidade

A análise de qualidade da camada Bronze permitiu avaliar completude, unicidade, consistência, acurácia e presença de valores extremos no conjunto de dados.

### Principais resultados

- Não foram identificados valores `NULL` ou campos vazios.
- Foram encontrados marcadores textuais de ausência de informação, como `IGNORADO`, `NÃO INFORMADO`, `NA` e `N/A`.
- Não foram identificados IDs duplicados ou registros completamente duplicados.
- As datas apresentaram formato consistente (`yyyy-MM-dd`) e os horários puderam ser interpretados corretamente.
- Os campos geográficos e rodoviários analisados apresentaram valores válidos.
- Os campos numéricos puderam ser convertidos sem perda de registros e não apresentaram valores negativos ou inconsistências lógicas básicas.
- Foi identificada uma inconsistência de capitalização na coluna `causa_acidente`, entre `Transitar no Acostamento` e `Transitar no acostamento`.
- A categoria genérica `Colisão lateral` aparece apenas nos meses de janeiro e fevereiro de 2021, indicando possível mudança ou refinamento do padrão de classificação da fonte.
- A coluna `tracado_via` possui estrutura multivalorada: 67.741 registros apresentam mais de uma característica separada por ponto e vírgula, correspondendo a aproximadamente 19,77% da base.
- Foram identificados valores extremos em variáveis como número de pessoas e veículos envolvidos, porém eles não serão removidos automaticamente, pois podem representar ocorrências reais de grande porte.
- Casos com quantidade de veículos superior à quantidade de pessoas também serão preservados, pois a própria natureza do registro pode gerar sub-registro de pessoas, principalmente ilesas.

### Decisão para a camada Silver

A camada Silver será responsável por:

- converter os campos para os tipos de dados adequados;
- padronizar categorias textuais comprovadamente equivalentes;
- padronizar marcadores de ausência de informação;
- preservar valores extremos que não apresentem evidência objetiva de erro;
- preservar a rastreabilidade dos dados originais;
- criar campos derivados necessários para as análises posteriores.